# Multi-Institution ISS "Conta de Gerência" Pipeline

This notebook is the notebook version of `multi_institution_pipeline.py`,
split into explained steps, with a final **automated summary generation**
section that turns the KPI table into plain-language findings per institution.

### What this pipeline does
Given a **folder of PDFs** it:

1. Discovers every PDF in the folder — no hardcoded file names or counts.
2. Extracts **institution-level metadata** (name, NISS, NIF, concelho) from
   page 1 of each PDF — transparently OCR'ing pages that have no text layer
   (scanned PDFs).
3. Walks every page and classifies it as `beneficiary` (one "Mapa A" page per
   Resposta Social / Atividade), `aggregate` (the institution-wide total Mapa
   A page), `comparticipacoes` (the ISS,IP funding page), or `other`
   (Balanço, Fluxos de Caixa, etc. — not needed for these KPIs).
4. Parses each beneficiary's income statement (label → this-year/prior-year
   value), re-zipping pdfplumber's multi-line cells back into a clean dict.
   Works the same whether the filing is for **2025 or 2024** (or any other
   year) — each record is tagged with its own filing year from page 1.
5. Pulls the **authoritative ISS,IP funding figure** per beneficiary from the
   dedicated "Mapa Comparticipações ISS, IP" page — not from the Mapa A
   income statement — because testing against two real institutions showed
   ISS,IP funding is booked under *different* P&L lines depending on the
   institution. This page's beneficiary names come back **truncated**, so the
   join to Mapa A uses prefix matching (the one genuinely fuzzy-matching step
   in the whole pipeline).
6. Cross-validates each institution's beneficiary sums against its own
   aggregate page — run on the *complete, unfiltered* set of activities.
7. **Filters down to elderly-care and childcare activities only**
   (Estrutura Residencial para Pessoas Idosas, Serviço de Apoio Domiciliário,
   Centro de Dia, Creche, Estabelecimento de Educação Pré-Escolar).
8. Computes the requested KPIs across **all institutions, all years, and all
   matching beneficiaries** in one combined table.
9. **Generates a plain-language summary**


## 1. Imports & configuration

Same as the script version. `PDF_FOLDER` should point at a directory
containing one PDF per institution.


In [1]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pdfplumber

try:
    import pytesseract
except ImportError:
    pytesseract = None
    warnings.warn(
        "pytesseract not installed - scanned PDFs (no text layer) will be "
        "skipped instead of OCR'd. Install with `pip install pytesseract`, "
        "and make sure the system OCR engine + Portuguese language pack are "
        "installed too (e.g. `apt-get install tesseract-ocr tesseract-ocr-por`)."
    )

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

PDF_FOLDER = Path("PDF_files")  # <-- folder containing all institution PDFs (any mix of years)

TABLE_SETTINGS = {
    "vertical_strategy": "lines",
    "horizontal_strategy": "lines",
    "snap_tolerance": 3,
    "join_tolerance": 3,
    "edge_min_length": 3,
}

# OCR settings — only used for pages that have no text layer (scanned PDFs)
OCR_RESOLUTION = 300   # dpi - higher = more accurate but slower
OCR_LANG = "por"       # Portuguese Tesseract language pack


## 2. Portuguese number parsing

Report numbers use `.` as thousands separator and `,` as decimal separator
(e.g. `276.569,56`). This converts such strings to `float`.


In [2]:
def normalize_pt_number(value_str):
    """Convert '1.234,56' / '-24,07' -> float. Returns np.nan if unparsable."""
    if value_str is None:
        return np.nan
    value_str = str(value_str).strip()
    if value_str == "":
        return np.nan
    negative = value_str.startswith("-")
    value_str = value_str.lstrip("-").replace(".", "").replace(",", ".")
    try:
        number = float(value_str)
    except ValueError:
        return np.nan
    return -number if negative else number


## 3. Institution-level metadata (page 1 of every PDF)

Each PDF's cover page carries the institution's identity (name, NISS, NIF,
concelho). We regex this once per file so every beneficiary row downstream
can be tagged with which institution it belongs to.


In [3]:
INSTITUTION_PATTERNS = {
    "ano": r"Ano:\s*(\d{4})",
    "nome_instituicao": r"Nome:\s*(.+)",              # first "Nome:" hit = institution
    "niss": r"NISS:\s*(\d+)",
    "concelho": r"Concelho:\s*(.+)",
}


def extract_institution_metadata(first_page_text):
    meta = {}
    for key, pattern in INSTITUTION_PATTERNS.items():
        match = re.search(pattern, first_page_text)
        meta[key] = match.group(1).strip() if match else None
    return meta


## 4. Beneficiary header metadata + page classification

Each Mapa A page carries free-text header fields (beneficiary name, avg.
users/staff, months active) above the actual table. `classify_page()`
determines what kind of page we're looking at, since institutions differ in
how many beneficiary pages they have and whether a `comparticipacoes` page is
present.


In [4]:
HEADER_PATTERNS = {
    "equipamento": r"Equipamento:\s*(.+)",
    "resposta_social": r"Resposta Social/Atividade:\s*(.+)",
    "n_medio_utentes": r"Nº Médio de Utentes:\s*([\d\.,]+)",
    "n_medio_funcionarios": r"Nº Médio de Funcionários:\s*([\d\.,]+)",
    "n_meses": r"Nº Meses:\s*(\d+)",
    "tipo_acordo": r"Tipo de Acordo:\s*(.+?)(?:\s{2,}|\n|$)",
}


def extract_header_metadata(text):
    header = {}
    for key, pattern in HEADER_PATTERNS.items():
        match = re.search(pattern, text)
        header[key] = match.group(1).strip() if match else None
    return header


def classify_page(text):
    """'beneficiary' (one Mapa A page per RS/Atividade), 'aggregate'
    (institution-wide Mapa A total), 'comparticipacoes' (authoritative ISS,IP
    funding per beneficiary), or 'other' (Balanço, Fluxos, etc.)."""
    if "Mapa Comparticipações ISS" in text:
        return "comparticipacoes"
    if "Número RS/Atividades agregadas" in text:
        return "aggregate"
    if "Resposta Social/Atividade:" in text:
        return "beneficiary"
    return "other"


## 4b. OCR fallback for scanned pages

Some institutions submit PDFs that are just a flattened scan/photo of the
printed form — no text layer at all (`page.extract_text()` returns nothing).

These two functions are drop-in replacements for `page.extract_text()` and
`page.extract_tables()`: they use the real text layer when it exists, and
transparently fall back to Tesseract OCR when it doesn't. For tables, cell
*boundaries* still come from the page's real vector gridlines (via
`find_tables`) — only the text inside each cell is filled in with OCR, so
numbers stay aligned to the correct row/column instead of OCR'ing the whole
page as one unstructured blob.

Requires, on your machine (not just in this notebook's Python environment):
```
apt-get install tesseract-ocr tesseract-ocr-por   # system OCR engine + PT language pack
pip install pytesseract
```

OCR output is noisier than a native text layer, so every record downstream
carries an `is_ocr` flag — worth a quick manual spot-check on those rows
rather than trusting them blindly.


In [5]:
def get_page_text(page):
    """Return (text, is_ocr) for a page: the native text layer if present,
    otherwise a whole-page OCR fallback (used for institution/header regex
    matching, where exact cell alignment doesn't matter)."""
    text = page.extract_text() or ""
    if text.strip() or pytesseract is None:
        return text, False
    image = page.to_image(resolution=OCR_RESOLUTION).original
    return pytesseract.image_to_string(image, lang=OCR_LANG), True


def get_page_tables(page, table_settings):
    """Return (tables, is_ocr) in the same list-of-rows-of-cells shape as
    page.extract_tables(). Falls back to per-cell OCR when the page has no
    text layer: cell boundaries still come from the page's real vector
    gridlines (via find_tables) - only the text inside each cell is filled
    in with Tesseract instead of pdfplumber's (empty) text extraction."""
    tables = page.extract_tables(table_settings=table_settings)
    has_text = any(cell for table in tables for row in table for cell in row if cell)
    if has_text or pytesseract is None:
        return tables, False

    found_tables = page.find_tables(table_settings=table_settings)
    if not found_tables:
        return [], False

    image = page.to_image(resolution=OCR_RESOLUTION).original
    scale = OCR_RESOLUTION / 72  # pdfplumber coordinates are in points (1/72")

    ocr_tables = []
    for table in found_tables:
        ocr_rows = []
        for row in table.rows:
            ocr_row = []
            for cell_bbox in row.cells:
                if cell_bbox is None:
                    ocr_row.append(None)
                    continue
                x0, top, x1, bottom = cell_bbox
                crop = image.crop((x0 * scale, top * scale, x1 * scale, bottom * scale))
                cell_text = pytesseract.image_to_string(
                    crop, lang=OCR_LANG, config="--psm 6"
                ).strip()
                ocr_row.append(cell_text or None)
            ocr_rows.append(ocr_row)
        ocr_tables.append(ocr_rows)
    return ocr_tables, True


## 5. Mapa A income-statement table parsing

pdfplumber merges every multi-line block of P&L labels into a single cell,
and does the same for the 2025/2024 value columns. Since labels and values
come out in the same top-to-bottom order, we reconstruct the mapping by
concatenating all label-lines and all value-lines (across every row of the
table) and zipping them together. This works whether or not the `NOTAS`
column (index 1) is populated — we never read it.


In [6]:
def extract_income_statement(table):
    """Return {label: (value_year1, value_year2)} for one Mapa A table.

    Value columns are taken as the *last two* columns of the row rather than
    a hardcoded row[2]/row[3], because some pages (notably OCR'd scans where
    the NOTAS divider line isn't always detected) come back with 3 columns
    instead of the usual 4 (label, NOTAS, year1, year2). Taking the last two
    columns works for both layouts.
    """
    labels, values_year1, values_year2 = [], [], []
    for row in table[2:]:
        label_cell = row[0] if len(row) > 0 else None
        val1_cell = row[-2] if len(row) >= 2 else None
        val2_cell = row[-1] if len(row) >= 1 else None
        labels.extend(label_cell.split("\n") if label_cell else [])
        values_year1.extend(val1_cell.split("\n") if val1_cell else [])
        values_year2.extend(val2_cell.split("\n") if val2_cell else [])

    n = min(len(labels), len(values_year1), len(values_year2))
    if n < len(labels):
        warnings.warn(
            f"Label/value count mismatch (labels={len(labels)}, "
            f"values={len(values_year1)}) - truncating to shortest."
        )

    return {
        labels[i].strip(): (normalize_pt_number(values_year1[i]), normalize_pt_number(values_year2[i]))
        for i in range(n)
    }


### 5b. Column-year sanity check (needed once 2024 files are mixed in)

`extract_income_statement` assumes the table's first value column is always
*this filing's own year* and the second is the prior year — true across
every sample seen so far, and it's exactly what lets a 2024 filing's "first
column" correctly mean 2024 without any extra code (each record is already
tagged with its own filing year from page 1's `Ano:` field).

This helper is a best-effort cross-check: if the table's literal
`2025`/`2024`-style header text is readable, it's compared against the year
on page 1. If they ever disagree, a warning is raised — so a silently
swapped column order gets caught instead of quietly poisoning the KPIs.


In [7]:
def detect_column_years(table):
    """Best-effort read of the literal year labels (e.g. '2025', '2024')
    from a Mapa A table's header row. Returns (year_col0, year_col1) as
    strings, or (None, None) if they can't be confidently detected. Uses the
    last two columns, same reasoning as extract_income_statement."""
    if len(table) < 2:
        return None, None
    header_row = table[1]
    if len(header_row) < 2:
        return None, None
    y0 = (header_row[-2] or "").strip()
    y1 = (header_row[-1] or "").strip()
    if re.fullmatch(r"\d{4}", y0) and re.fullmatch(r"\d{4}", y1):
        return y0, y1
    return None, None


## 6. Whitelisted P&L line items

Only the ~10 top-level line items needed for the KPIs are pulled — several
labels (e.g. `"ISS, IP"`) repeat at different nesting depths on the same
page, so pulling everything would collide.


In [8]:
LINE_ITEMS = {
    "vendas": "Vendas",
    "servicos_prestados": "Serviços prestados",
    "servicos_prestados_particulares": "Serviços prestados - Particulares",
    "subsidios_publicos": "Subsídios, doações e legados à exploração",
    "iss_ip":"ISS, IP",
    "iss_ip_2": "ISS, IP – Apoios excecionais e extraordinários",
    "outros_rendimentos": "Outros rendimentos",
    "custo_mercadorias": "Custo das mercadorias vendidas e das matérias consumidas",
    "fse": "Fornecimentos e serviços externos",
    "gastos_pessoal": "Gastos com pessoal",
    "outros_gastos": "Outros gastos",
    "ebitda": "Resultado antes de depreciações, gastos de financiamento e impostos",
    # Kept for reference if needed for KPIs
    "depreciacao": "Gastos/reversões de depreciação e de amortização",
    "resultado_liquido": "Resultado líquido do período",
}


## 7. Authoritative ISS,IP funding (cross-institution finding)

Testing against two real institutions showed ISS,IP funding is **not**
always booked in the same P&L line:

- Some institutions book it under `"Subsídios de entidades públicas"`
- Others book it under `"Serviços prestados - Entidades Públicas → ISS, IP"`

Relying on a single fixed Mapa A line silently under- or over-counts it. The
report has a dedicated **"Mapa Comparticipações ISS, IP"** page whose
`Total da Comparticipação` column is authoritative regardless of booking
choice — so we use that instead.

Two wrinkles:
- pdfplumber's line-based `extract_tables()` doesn't reliably detect row
  borders on this specific page (only the header row comes back), so we
  regex-parse the page's plain text instead.
- That text **truncates** long beneficiary names (fixed-width column), e.g.
  `"SERVIÇO DE APOIO DOMICILIÁ"` instead of `"...DOMICILIÁRIO"` — so joining
  it back to the Mapa A beneficiary name needs **prefix matching**, not exact
  match. This is the one genuinely fuzzy-matching step in the pipeline.


In [9]:
NUM_PT = r"-?[\d.]+,\d{2}"
COMPARTICIPACOES_ROW_RE = re.compile(
    rf"^(?P<equipamento>\S+)\s+(?P<name>.+?)\s+(?P<tipo>Atividade|Resposta Soc)\s+"
    rf"(?P<utentes>{NUM_PT})\s+(?P<funcionarios>{NUM_PT})\s+"
    rf"(?P<conta72>{NUM_PT})\s+(?P<conta751>{NUM_PT})\s+"
    rf"(?P<conta751_apoios>{NUM_PT})\s+(?P<ap78>{NUM_PT})\s+"
    rf"(?P<an68>{NUM_PT})\s+(?P<total>{NUM_PT})\s+"
    rf"(?P<resultado>{NUM_PT})\s+(?P<gasto>{NUM_PT})$"
)


def extract_comparticipacoes(text):
    """Regex-parse the 'Mapa Comparticipações ISS, IP' page into one dict per
    beneficiary row: {truncated_name: total_comparticipacao}."""
    rows = {}
    for line in text.splitlines():
        match = COMPARTICIPACOES_ROW_RE.match(line.strip())
        if match:
            rows[match.group("name").strip()] = normalize_pt_number(match.group("total"))
    return rows


def strip_code_prefix(resposta_social_name):
    """'1103 - CRECHE' -> 'CRECHE'; 'CANTINA SOCIAL' -> 'CANTINA SOCIAL'."""
    if resposta_social_name is None:
        return None
    return re.sub(r"^\d+\s*-\s*", "", resposta_social_name).strip().upper()


def match_comparticipacao(resposta_social_name, comparticipacoes_by_name):
    """Prefix-match a Mapa A beneficiary name against the (possibly truncated)
    names parsed from the Mapa Comparticipações page."""
    target = strip_code_prefix(resposta_social_name)
    if target is None:
        return np.nan, None

    matches = [
        (key, value) for key, value in comparticipacoes_by_name.items()
        if target.startswith(key.upper()) or key.upper().startswith(target)
    ]
    if len(matches) == 1:
        return matches[0][1], matches[0][0]
    if len(matches) > 1:
        warnings.warn(f"Ambiguous comparticipação match for '{resposta_social_name}': {matches}")
    return np.nan, None


## 7b. Activity filter — keep only elderly-care and childcare responses

Each institution can report many `Resposta Social/Atividade` lines (canteens,
volunteering programmes, etc.) that aren't relevant to this analysis. Keep
only:

- **Elderly care**: Estrutura Residencial para Pessoas Idosas (ERPI),
  Serviço de Apoio Domiciliário (SAD), Centro de Dia
- **Childcare / nursery**: Creche, Estabelecimento de Educação Pré-Escolar

Matching is done on the *stripped* name (via `strip_code_prefix`, e.g.
`'2101 - CENTRO DE DIA'` → `'CENTRO DE DIA'`) with loose substring matching,
since some institutions phrase these slightly differently.


In [10]:
ALLOWED_ACTIVITIES = [
    "ESTRUTURA RESIDENCIAL PARA PESSOAS IDOSAS",
    "SERVIÇO DE APOIO DOMICILIÁRIO",
    "CENTRO DE DIA",
    "CRECHE",
    "ESTABELECIMENTO DE EDUCAÇÃO PRÉ-ESCOLAR",
]

ACTIVITY_GROUP = {
    "ESTRUTURA RESIDENCIAL PARA PESSOAS IDOSAS": "Idosos",
    "SERVIÇO DE APOIO DOMICILIÁRIO": "Idosos",
    "CENTRO DE DIA": "Idosos",
    "CRECHE": "Infância",
    "ESTABELECIMENTO DE EDUCAÇÃO PRÉ-ESCOLAR": "Infância",
}


def matched_allowed_activity(resposta_social_name):
    """Return the canonical ALLOWED_ACTIVITIES entry this beneficiary matches,
    or None if it's outside the elderly-care/childcare scope."""
    name = strip_code_prefix(resposta_social_name)
    if name is None:
        return None
    for allowed in ALLOWED_ACTIVITIES:
        if allowed in name or name in allowed:
            return allowed
    return None


def is_allowed_activity(resposta_social_name):
    return matched_allowed_activity(resposta_social_name) is not None


## 8. Per-PDF extraction

Ties everything above together for a single institution's PDF: reads page 1
for institution metadata, classifies every other page, parses beneficiary and
aggregate Mapa A tables, captures the comparticipações page, then joins the
authoritative ISS,IP funding onto each beneficiary record.


In [11]:
def process_pdf(pdf_path):
    """Return (beneficiary_records, aggregate_record, institution_meta)
    for one institution's PDF."""

    beneficiary_records = []
    aggregate_record = None
    comparticipacoes_by_name = {}
    institution_meta = {"source_file": pdf_path.name}

    with pdfplumber.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf.pages, start=1):
            text, is_ocr_text = get_page_text(page)

            if page_number == 1:
                institution_meta.update(extract_institution_metadata(text))
                continue  # page 1 has no Mapa A table

            page_type = classify_page(text)

            if page_type == "comparticipacoes":
                comparticipacoes_by_name = extract_comparticipacoes(text)
                continue

            if page_type not in ("beneficiary", "aggregate"):
                continue

            tables, is_ocr_tables = get_page_tables(page, TABLE_SETTINGS)
            if not tables:
                continue

            pnl = extract_income_statement(tables[0])
            header = extract_header_metadata(text)

            # Sanity-check the assumed "col 0 = this filing's own year" order
            # now that 2024 (and other-year) filings are mixed in with 2025 ones.
            #year_col0, year_col1 = detect_column_years(tables[0])
            #if year_col0 and year_col0 != institution_meta.get("ano"):
                #warnings.warn(
                    #f"{pdf_path.name} p.{page_number}: table header shows "
                    #f"{year_col0}/{year_col1} but page 1 'Ano' is "
                    #f"{institution_meta.get('ano')} - check column order."
                #)

            record = {
                "source_file": pdf_path.name,
                "page": page_number,
                "resposta_social": header["resposta_social"],
                "equipamento": header["equipamento"],
                "tipo_acordo": header["tipo_acordo"],
                "n_medio_utentes": normalize_pt_number(header["n_medio_utentes"]),
                "n_medio_funcionarios": normalize_pt_number(header["n_medio_funcionarios"]),
                "n_meses": float(header["n_meses"]) if header["n_meses"] else 12.0,
                "is_ocr": is_ocr_text or is_ocr_tables,
            }
            for col, label in LINE_ITEMS.items():
                record[col] = pnl.get(label, (np.nan, np.nan))[0]  # index 0 = this filing's own year

            if page_type == "beneficiary":
                beneficiary_records.append(record)
            else:  # aggregate
                aggregate_record = record

    # Join authoritative ISS,IP funding onto each beneficiary via prefix matching.
    for record in beneficiary_records:
        funding, matched_key = match_comparticipacao(
            record["resposta_social"], comparticipacoes_by_name
        )
        record["ss_funding_authoritative"] = funding
        record["_comparticipacoes_matched_key"] = matched_key

    return beneficiary_records, aggregate_record, institution_meta


## 9. Cross-institution validation

Sums the whitelisted line items across an institution's beneficiary pages and
compares them to that institution's own aggregate Mapa A page — a free
correctness check with no external ground truth needed.


In [12]:
def validate_institution(beneficiary_records, aggregate_record, institution_name):
    if aggregate_record is None:
        print(f"  [{institution_name}] No aggregate page found - skipping validation.")
        return True

    mismatches = []
    for key in LINE_ITEMS:
        summed = sum(r.get(key) or 0 for r in beneficiary_records)
        reported = aggregate_record.get(key)
        if reported is not None and abs(summed - reported) > 0.01:
            mismatches.append((key, summed, reported))

    if mismatches:
        print(f"  ⚠ [{institution_name}] Mismatches vs aggregate page:")
        for key, summed, reported in mismatches:
            print(f"      {key}: sum={summed:.2f} vs aggregate={reported:.2f}")
        return False

    print(f"  ✓ [{institution_name}] Beneficiary sums match the aggregate page.")
    return True


## 10. KPI computation

Same 10 KPIs as before, now vectorised across every institution/beneficiary
row at once. Social-security funding uses `ss_funding_authoritative` (falling
back to `subsidios_publicos` only if no comparticipações match was found),
per the finding in Section 7.


In [13]:
def compute_kpis(df):
    df = df.copy()

    df["ss_funding"] = df["ss_funding_authoritative"].fillna(df["subsidios_publicos"])

    df["revenue"] = df["servicos_prestados"].fillna(0) + df["subsidios_publicos"].fillna(0) +df["vendas"].fillna(0) + df["outros_rendimentos"].fillna(0)
    df["total_cost"] = -(
        df["custo_mercadorias"].fillna(0)
        + df["fse"].fillna(0)
        + df["gastos_pessoal"].fillna(0)
        + df["outros_gastos"].fillna(0)
    )
    df["labour_cost"] = -df["gastos_pessoal"].fillna(0)

    df["monthly_revenue"] = df["revenue"] / df["n_meses"] 
    df["monthly_fee"] = df["servicos_prestados"] / df["n_meses"]
    df["monthly_social_security_funding"] = df["ss_funding"] / df["n_meses"]
    df["monthly_cost"] = df["total_cost"] / df["n_meses"]
    df["monthly_ebitda"] = df["ebitda"] / df["n_meses"] 
    df["monthly_labour_cost"] = df["labour_cost"] / df["n_meses"]

    df["monthly_revenue_per_beneficiary"] = df["monthly_revenue"] / df["n_medio_utentes"]
    df["monthly_fee_per_beneficiary"] = df["servicos_prestados_particulares"] / df["n_medio_utentes"] / df["n_meses"]
    df["monthly_social_security_funding_per_beneficiary"] = (df["iss_ip"] + df["iss_ip_2"]) / df["n_meses"] / df["n_medio_utentes"]
    df["monthly_cost_per_beneficiary"] = df["monthly_cost"] / df["n_medio_utentes"]
    df["monthly_ebitda_per_beneficiary"] = df["monthly_revenue_per_beneficiary"] - df["monthly_cost_per_beneficiary"]


    df["monthly_revenue_per_worker"] = df["monthly_revenue"] / df["n_medio_funcionarios"]
    df["monthly_cost_per_worker"] = df["monthly_cost"] / df["n_medio_funcionarios"]
    df["monthly_labour_cost_per_worker"] = df["monthly_labour_cost"] / df["n_medio_funcionarios"]

    df["beneficiary_worker_ratio"] = df["n_medio_utentes"] / df["n_medio_funcionarios"]
    df["ss_funding_cost_coverage_ratio"] = df["monthly_social_security_funding"] / df["monthly_cost_per_beneficiary"]

    return df


KPI_COLS = [
    "institution_id", "ano", "resposta_social", "activity_group",
    "n_medio_utentes", "n_medio_funcionarios",
    "monthly_revenue_per_beneficiary", "monthly_fee_per_beneficiary",
    "monthly_social_security_funding_per_beneficiary", "monthly_cost_per_beneficiary","monthly_ebitda_per_beneficiary",
    "monthly_revenue_per_worker", "monthly_cost_per_worker", "monthly_labour_cost_per_worker",
    "beneficiary_worker_ratio", "ss_funding_cost_coverage_ratio",
    
]


## 11. Batch driver: run over every PDF in the folder


In [14]:
def run_batch(pdf_folder):
    pdf_paths = sorted(Path(pdf_folder).glob("*.pdf"))
    if not pdf_paths:
        raise SystemExit(f"No PDFs found in {pdf_folder}")

    all_beneficiary_rows = []
    validation_results = {}
    print(f"Found {len(pdf_paths)} PDF(s) to process.\n")

    for pdf_path in pdf_paths:
        print(f"Processing {pdf_path.name} ...")
        beneficiary_records, aggregate_record, institution_meta = process_pdf(pdf_path)

        institution_name = institution_meta.get("nome_instituicao") or pdf_path.name
        validation_results[institution_name] = validate_institution(
            beneficiary_records, aggregate_record, institution_name
        )

        for record in beneficiary_records:
            record.update({
                "niss": institution_meta.get("niss"),
                "nome_instituicao": institution_meta.get("nome_instituicao"),
                "concelho": institution_meta.get("concelho"),
                "centro_distrital": institution_meta.get("centro_distrital"),
                "ano": institution_meta.get("ano"),
            })
            all_beneficiary_rows.append(record)

        print(f"  -> {len(beneficiary_records)} beneficiaries parsed "
              f"(validation above runs on ALL activities, before filtering).\n")

    combined = pd.DataFrame(all_beneficiary_rows)

    # Keep only elderly-care (ERPI/SAD/Centro de Dia) and childcare
    # (Creche/Pré-Escolar) beneficiaries. Filtering happens *after*
    # validate_institution() so the aggregate-page cross-check still runs
    # against each institution's complete, unfiltered set of activities.
    n_before = len(combined)
    combined["activity_group"] = combined["resposta_social"].apply(
        lambda name: ACTIVITY_GROUP.get(matched_allowed_activity(name))
    )
    combined = combined[combined["resposta_social"].apply(is_allowed_activity)].reset_index(drop=True)
    print(f"Kept {len(combined)}/{n_before} beneficiary rows after the "
          f"elderly-care/childcare activity filter.\n")

    kpi_df = compute_kpis(combined)
    return combined, kpi_df, validation_results
import hashlib

def anonymize_institutions(df, niss_col="niss", name_col="nome_instituicao"):
    """Replace NISS + institution name with a stable pseudonymous ID.
    Same institution -> same ID across runs (derived from NISS hash),
    but the original NISS/name are not retained in the output."""
    df = df.copy()

    def pseudo_id(niss):
        if pd.isna(niss):
            return "INST_UNKNOWN"
        digest = hashlib.sha256(str(niss).encode()).hexdigest()[:8]
        return f"INST_{digest}"

    df["institution_id"] = df[niss_col].apply(pseudo_id)
    df = df.drop(columns=[niss_col, name_col])
    return df

raw_df, kpi_df, validation_results = run_batch(PDF_FOLDER)
raw_df = anonymize_institutions(raw_df)
kpi_df = anonymize_institutions(kpi_df)
kpi_table = kpi_df[KPI_COLS].round(2)   # rebuild — old kpi_table is stale


Found 23 PDF(s) to process.

Processing DOC 2.pdf ...
  ✓ [CENTRO SOCIAL PAROQUIAL DE NOSSA SENHORA DA CONCEIÇÃO] Beneficiary sums match the aggregate page.
  -> 3 beneficiaries parsed (validation above runs on ALL activities, before filtering).

Processing DOC 3.pdf ...
  ✓ [LIGA DE SOLIDARIEDADE SOCIAL E MELHORAMENTOS - OS AMIGOS DE ALBARDO] Beneficiary sums match the aggregate page.
  -> 1 beneficiaries parsed (validation above runs on ALL activities, before filtering).

Processing DOC 7.pdf ...
  ✓ [CENTRO SOCIAL PAROQUIAL DE FREIXIANDA] Beneficiary sums match the aggregate page.
  -> 6 beneficiaries parsed (validation above runs on ALL activities, before filtering).

Processing OCIP 1_24.pdf ...
  ✓ [CENTRO SOCIAL PAROQUIA GUALTAR] Beneficiary sums match the aggregate page.
  -> 4 beneficiaries parsed (validation above runs on ALL activities, before filtering).

Processing OCIP 2_24.pdf ...
  ✓ [CENTRO PAROQUIAL DE BEM-ESTAR SOCIAL DE RIO MAIOR] Beneficiary sums match the aggregat

## 12. Inspect the combined raw data and KPI table


In [15]:
raw_df


,source_file,page,resposta_social,equipamento,tipo_acordo,n_medio_utentes,n_medio_funcionarios,n_meses,is_ocr,vendas,servicos_prestados,servicos_prestados_particulares,subsidios_publicos,iss_ip,iss_ip_2,outros_rendimentos,custo_mercadorias,fse,gastos_pessoal,outros_gastos,ebitda,depreciacao,resultado_liquido,ss_funding_authoritative,_comparticipacoes_matched_key,concelho,centro_distrital,ano,activity_group,institution_id
0,DOC 2.pdf,2,1103 - CRECHE,1 - SEDE,Típico Tipo de Atividade:,39.00,8.25,12.00,False,0.00,"8,719.00","8,719.00","276,569.56","276,569.56",0.00,"15,870.89","-24,684.07","-27,995.84","-167,486.01","-39,829.63","41,163.90","-3,934.24","37,229.66","276,569.56",CRECHE,GUIMARÃES,None,2025,Infância,INST_33c54ad8
1,DOC 2.pdf,3,1104 - ESTABELECIMENTO DE EDUCAÇÃO PRÉ-ESCOLAR,1 - SEDE,Típico Tipo de Atividade:,40.00,6.25,12.00,False,0.00,"56,181.11","56,181.11","139,124.15","139,124.15",0.00,"15,759.49","-26,089.61","-28,910.45","-156,609.75","-45,126.31","-45,671.37","-4,034.09","-49,705.46","139,124.15",ESTABELECIMENTO DE EDUC,GUIMARÃES,None,2025,Infância,INST_33c54ad8
2,DOC 2.pdf,4,2103 - CENTRO DE DIA,1 - SEDE,Típico Tipo de Atividade:,20.00,5.50,12.00,False,0.00,"51,412.11","51,412.11","50,354.98","50,354.98",0.00,"9,023.98","-16,258.46","-22,064.55","-90,539.90","-20,521.97","-38,593.81","-2,017.04","-40,610.85","50,354.98",CENTRO DE DIA,GUIMARÃES,None,2025,Idosos,INST_33c54ad8
3,DOC 3.pdf,2,2101 - SERVIÇO DE APOIO DOMICILIÁRIO,1 - SEDE,Típico Tipo de Atividade:,14.00,4.00,12.00,False,0.00,"28,115.50","26,902.50","71,344.25","66,969.25",0.00,"1,434.99","-21,826.29","-12,624.79","-63,022.35",-433.86,"2,987.45","-4,344.27","-1,373.55","66,969.25",SERVIÇO DE APOIO DOMICILIÁ,GUARDA,None,2025,Idosos,INST_ed1ce449
4,DOC 7.pdf,2,1103 - CRECHE,1 - SEDE,Típico Tipo de Atividade:,48.00,13.00,12.00,False,0.00,"325,419.40",0.00,0.00,0.00,0.00,"19,393.34","-58,274.74","-29,946.59","-173,181.76",-53.61,"83,356.04","-23,732.17","59,623.87","325,419.40",CRECHE,OURÉM,None,2025,Infância,INST_9c027b3d
5,DOC 7.pdf,3,2101 - SERVIÇO DE APOIO DOMICILIÁRIO,1 - SEDE,Típico Tipo de Atividade:,15.00,5.00,12.00,False,0.00,"153,521.05","49,531.53",0.00,0.00,0.00,"5,565.77","-41,083.13","-16,720.22","-96,133.46",0.00,"5,150.01","-5,048.47",101.54,"103,989.52",SERVIÇO DE APOIO DOMICILIÁ,OURÉM,None,2025,Idosos,INST_9c027b3d
6,DOC 7.pdf,4,2103 - CENTRO DE DIA,1 - SEDE,Típico Tipo de Atividade:,6.00,3.00,12.00,False,0.00,"24,906.29","11,667.47",0.00,0.00,0.00,"2,343.32","-10,297.98","-39,300.53","-56,279.19",0.00,"-78,628.09","-7,308.72","-85,936.81","13,238.82",CENTRO DE DIA,OURÉM,None,2025,Idosos,INST_9c027b3d
7,DOC 7.pdf,5,2107 - ESTRUTURA RESIDENCIAL PARA PESSOAS IDOSAS,1 - SEDE,Típico Tipo de Atividade:,45.00,24.00,12.00,False,0.00,"721,608.32","383,477.20","13,127.47",0.00,0.00,"82,493.62","-75,912.47","-130,844.12","-537,257.10",-664.66,"72,551.06","-101,327.81","-82,045.16","338,131.12",ESTRUTURA RESIDENCIAL PA,OURÉM,None,2025,Idosos,INST_9c027b3d
8,OCIP 1_24.pdf,2,1103 - CRECHE,1 - SEDE,Típico Tipo de Atividade:,66.00,16.00,12.00,False,0.00,"17,981.14","17,981.14","515,406.22","515,406.22",0.00,"14,357.43","-36,876.69","-80,217.61","-358,118.77","-3,195.53","69,336.19","-19,108.99","58,447.24",NaN,NaN,BRAGA,None,2024,Infância,INST_78679529
9,OCIP 1_24.pdf,3,1104 - ESTABELECIMENTO DE EDUCAÇÃO PRÉ-ESCOLAR,1 - SEDE,Típico Tipo de Atividade:,60.00,7.00,12.00,False,0.00,"132,858.89","132,858.89","139,825.51","139,825.51",0.00,"7,331.40","-18,830.64","-45,311.49","-184,886.08","-1,631.77","29,355.82","-9,757.77","23,795.52",NaN,NaN,BRAGA,None,2024,Infância,INST_78679529


In [16]:
kpi_table


,institution_id,ano,resposta_social,activity_group,n_medio_utentes,n_medio_funcionarios,monthly_revenue_per_beneficiary,monthly_fee_per_beneficiary,monthly_social_security_funding_per_beneficiary,monthly_cost_per_beneficiary,monthly_ebitda_per_beneficiary,monthly_revenue_per_worker,monthly_cost_per_worker,monthly_labour_cost_per_worker,beneficiary_worker_ratio,ss_funding_cost_coverage_ratio
0,INST_33c54ad8,2025,1103 - CRECHE,Infância,39.00,8.25,643.50,18.63,590.96,555.55,87.96,"3,042.01","2,626.22","1,691.78",4.73,41.49
1,INST_33c54ad8,2025,1104 - ESTABELECIMENTO DE EDUCAÇÃO PRÉ-ESCOLAR,Infância,40.00,6.25,439.72,117.04,289.84,534.87,-95.15,"2,814.20","3,423.15","2,088.13",6.40,21.68
2,INST_33c54ad8,2025,2103 - CENTRO DE DIA,Idosos,20.00,5.50,461.63,214.22,209.81,622.44,-160.81,"1,678.65","2,263.41","1,371.82",3.64,6.74
3,INST_ed1ce449,2025,2101 - SERVIÇO DE APOIO DOMICILIÁRIO,Idosos,14.00,4.00,600.56,160.13,398.63,582.78,17.78,"2,101.97","2,039.74","1,312.97",3.50,9.58
4,INST_9c027b3d,2025,1103 - CRECHE,Infância,48.00,13.00,598.63,0.00,0.00,453.92,144.72,"2,210.34","1,676.00","1,110.14",3.69,59.74
5,INST_9c027b3d,2025,2101 - SERVIÇO DE APOIO DOMICILIÁRIO,Idosos,15.00,5.00,883.82,275.18,0.00,855.20,28.61,"2,651.45","2,565.61","1,602.22",3.00,10.13
6,INST_9c027b3d,2025,2103 - CENTRO DE DIA,Idosos,6.00,3.00,378.47,162.05,0.00,"1,470.52","-1,092.06",756.93,"2,941.05","1,563.31",2.00,0.75
7,INST_9c027b3d,2025,2107 - ESTRUTURA RESIDENCIAL PARA PESSOAS IDOSAS,Idosos,45.00,24.00,"1,513.39",710.14,0.00,"1,379.03",134.35,"2,837.60","2,585.69","1,865.48",1.88,20.43
8,INST_78679529,2024,1103 - CRECHE,Infância,66.00,16.00,691.60,22.70,650.77,604.05,87.55,"2,852.84","2,491.71","1,865.20",4.12,71.10
9,INST_78679529,2024,1104 - ESTABELECIMENTO DE EDUCAÇÃO PRÉ-ESCOLAR,Infância,60.00,7.00,388.91,184.53,194.20,348.14,40.77,"3,333.52","2,984.05","2,201.02",8.57,33.47


## 13. LLM 


In [17]:
table_text = kpi_table.to_markdown(index=False)

In [18]:
from ollama import chat

MODEL_NAME = "hf.co/duarteocarmo/AMALIA-9B-0626-SFT-GGUF:Q4_K_M"

def ask_amalia(prompt):
    response = chat(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": (
                    "És um analista de dados especializado em KPIs. "
                    "Responde sempre em português europeu (pt-PT). "
                    "Analisa os dados cuidadosamente e não inventes valores "
                    "que não estejam presentes na tabela."
                ),
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
    )

    return response["message"]["content"]

In [19]:


prompt = f"""
Pode analisar a informação deste quadro:

{table_text}

e apresentar sugestões para negociar com o governo a obtenção de mais recursos?
"""

answer = ask_amalia(prompt)

print(answer)

KeyboardInterrupt: 

In [ ]:
from ollama import chat

MODEL_NAME = "llama3.2:latest"

def ask_amalia(prompt):
    response = chat(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a data analyst specializing in KPIs. "
                    "Analyze the data carefully and do not invent values ​​"
                    "that are not present in the table."
                    "The currenty is Euro (€)."


                ),
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
    )

    return response["message"]["content"]

In [ ]:
table_text = kpi_table.to_markdown(index=False)

prompt = f"""
Can you analyze below table and provide suggestions for negotiating with the government to obtain more resources?

{table_text}

"""

answer = ask_amalia(prompt)

print(answer)